Normalizar nombres y construir variables derivadas

En esta práctica trabajaremos con trabajar con dos tareas que suelen aparecer en una etapa avanzada de preparación: estandarizar nombres de columnas y crear nuevas columnas útiles para el análisis.

Ya trabajamos antes con nombres de columnas, así que no trabajaremos con tratar este tema como algo completamente nuevo. Esta vez lo retomaremos dentro de un flujo de limpieza más realista. La idea será justificar por qué resulta útil usar nombres consistentes y aplicar una estrategia simple para transformar columnas como `Transaction ID`, `Price Per Unit` o `Transaction Date` en nombres más cómodos para trabajar.

Luego trabajaremos con crear nuevas columnas a partir de datos ya existentes. Por ejemplo, si tenemos una columna de cantidad y otra de precio unitario, es posible calcular un importe total. También es posible crear columnas de validación para comparar un valor registrado con un valor calculado, o columnas derivadas de una fecha.

Estas nuevas columnas no son adornos. Muchas veces permiten responder preguntas que el dataset original no podía responder directamente, o ayudan a verificar la consistencia interna de los datos.

Al finalizar este notebook deberías poder:

- Comprender por qué resulta útil estandarizar nombres de columnas.
- Renombrar columnas usando un criterio uniforme.
- Verificar que los nuevos nombres quedaron correctamente aplicados.
- Crear columnas nuevas a partir de operaciones entre columnas existentes.
- Crear columnas de validación para comparar valores registrados y calculados.
- Crear columnas derivadas de una fecha.
- Comprender que crear columnas nuevas puede enriquecer el análisis y también ayudar a controlar la calidad de los datos.

### Punto de control

La consistencia de los nombres reduce errores al escribir transformaciones y facilita que otra persona entienda el flujo sin consultar cada línea de código.

## Fuente para la práctica

Para esta sección trabajaremos con volver al dataset real **Cafe Sales — Dirty Data for Cleaning Training**.

Este dataset nos permite trabajar con varias situaciones ya vistas: columnas con nombres que contienen espacios y mayúsculas, columnas numéricas que necesitan conversión, una columna de fecha y una relación interna entre cantidad, precio unitario e importe total.

Como venimos haciendo en esta parte del recorrido, trabajaremos con cargar el dataset desde la fuente local. En Google Colab, `cargador_colab` ya suele estar disponible, por lo que comenzamos directamente con las importaciones necesarias.

### Punto de control

Las columnas nuevas deben conservar una relación clara con sus variables de origen. Documentar esa relación ayuda a interpretar los resultados y a repetir el procedimiento.

In [ ]:
# Carga manual del dataset en Google Colab

from google.colab import files
import io
import pandas as pd

# El usuario selecciona el CSV desde su equipo.
uploaded = files.upload()
archivos = list(uploaded.keys())
archivos_csv = [archivo for archivo in archivos if archivo.lower().endswith('.csv')]

if not archivos_csv:
    raise ValueError('Debes subir al menos un archivo con extensión .csv')

nombre_csv = archivos_csv[0]
df = pd.read_csv(io.BytesIO(uploaded[nombre_csv]))

columnas_requeridas = {
    'Transaction ID', 'Item', 'Quantity', 'Price Per Unit',
    'Total Spent', 'Payment Method', 'Location', 'Transaction Date'
}
columnas_faltantes = columnas_requeridas - set(df.columns)
if columnas_faltantes:
    raise ValueError(
        'El archivo no corresponde al dataset esperado. '
        f'Faltan columnas: {sorted(columnas_faltantes)}'
    )

df.head()

La salida de `head()` nos permite confirmar que el archivo fue cargado correctamente.

En esta sección trabajaremos con trabajar sobre una copia del dataset. De esa manera es posible transformar nombres, convertir columnas y crear nuevas variables sin perder el punto de partida original.

### Punto de control

Trabajar sobre una copia mantiene disponible la fuente original. Así es posible comparar el estado inicial con la versión preparada y revisar cada decisión.

## Utilidad de nombres consistentes

Los nombres de columnas son una parte relevante de la estructura del dataset.

En el archivo original encontramos nombres como:

```text
Transaction ID
Price Per Unit
Total Spent
Payment Method
Transaction Date
```

Estos nombres son comprensibles, pero no siempre son los más cómodos para trabajar en código. Tienen espacios, usan mayúsculas y están escritos con un estilo más cercano a un encabezado de planilla que a un nombre de variable.

En Pandas es posible trabajar con nombres así sin problema, usando corchetes:

```python
df["Price Per Unit"]
```

Aun así, en un flujo de análisis largo puede ser más cómodo usar nombres consistentes, en minúsculas y con guiones bajos:

```python
df["price_per_unit"]
```

Estandarizar nombres no cambia los valores del dataset. Lo que cambia es la forma en que nos referimos a las columnas. Esto puede reducir errores de escritura, hacer que el código sea más legible y facilitar la aplicación de transformaciones encadenadas.

En esta etapa trabajaremos con adoptar un criterio simple:

```text
usar minúsculas
reemplazar espacios por guiones bajos
mantener nombres descriptivos
```

Ese criterio suele conocerse como estilo `snake_case`, porque las palabras se escriben en minúsculas y separadas por guiones bajos.

### Punto de control

Una conversión puede producir valores faltantes cuando encuentra textos especiales. Esos casos deben conservarse visibles para no confundir una ausencia con un dato válido.

In [ ]:
df.columns

El resultado permite observar los nombres originales de las columnas.

Antes de crear nuevas columnas o aplicar transformaciones más largas, trabajaremos con construir una copia del dataset y renombrar sus columnas con un criterio más uniforme.

### Punto de control

Las reglas de validación deben contemplar valores completos y casos insuficientes. No contar con todos los componentes no equivale necesariamente a que exista una inconsistencia.

## Aplicar una convención de nombres

Para estandarizar los nombres de columnas es posible usar `rename()`.

Para este ejemplo trabajaremos con crear una copia llamada `df_trabajo` y trabajaremos con renombrar las columnas principales usando nombres en minúsculas y con guiones bajos.

No estamos cambiando los datos. Solo estamos cambiando los nombres con los que trabajaremos con referirnos a cada columna.

### Punto de control

Las variables derivadas amplían las posibilidades de análisis, pero no sustituyen las columnas originales. Mantener ambas versiones permite rastrear cómo se obtuvo cada resultado.

In [ ]:
df_trabajo = df.copy()

df_trabajo = df_trabajo.rename(columns={
    "Transaction ID": "transaction_id",
    "Item": "item",
    "Quantity": "quantity",
    "Price Per Unit": "price_per_unit",
    "Total Spent": "total_spent",
    "Payment Method": "payment_method",
    "Location": "location",
    "Transaction Date": "transaction_date"
})

df_trabajo.head()

A continuación el `DataFrame` tiene nombres de columnas más cómodos para trabajar.

Podemos verificar la estructura revisando nuevamente los nombres:

### Punto de control

Antes de interpretar una nueva columna, conviene revisar sus tipos, ejemplos y valores frecuentes. Una inspección pequeña puede detectar errores antes de usarlos en cálculos posteriores.

In [ ]:
df_trabajo.columns

Los nombres quedaron en un formato más uniforme.

Este paso parece pequeño, pero puede facilitar mucho el trabajo posterior. Por ejemplo, al escribir transformaciones más largas, nombres como `price_per_unit` o `transaction_date` suelen ser más cómodos y menos propensos a errores que nombres con espacios y mayúsculas.

También prepara el terreno para crear nuevas columnas con nombres consistentes.

### Punto de control

La verificación final debe confirmar nombres, tipos, dimensiones y ejemplos. Un dataset preparado es aquel cuyo contenido y estructura pueden explicarse con claridad.

## Preparar variables numéricas

Antes de crear columnas nuevas a partir de operaciones matemáticas, es necesario asegurarnos de que las columnas involucradas sean realmente numéricas.

En nuestro dataset, las columnas `quantity`, `price_per_unit` y `total_spent` deberían representar cantidades o importes. Aun así, como vimos antes, el archivo original puede contener valores problemáticos como `"UNKNOWN"` o `"ERROR"`, y por eso Pandas puede haberlas cargado como texto.

Si intentamos multiplicar columnas que todavía están como texto, el resultado puede fallar o no representar un cálculo numérico real. Por esa razón, antes de crear una columna calculada, trabajaremos con convertir esas variables a formato numérico.

### Punto de control

La consistencia de los nombres reduce errores al escribir transformaciones y facilita que otra persona entienda el flujo sin consultar cada línea de código.

In [ ]:
columnas_numericas = [
    "quantity",
    "price_per_unit",
    "total_spent"
]

for columna in columnas_numericas:
    df_trabajo[columna] = pd.to_numeric(
        df_trabajo[columna],
        errors="coerce"
    )

df_trabajo[columnas_numericas].dtypes

A continuación las columnas seleccionadas fueron convertidas a tipo numérico.

Usamos `errors="coerce"` para que los valores que no puedan convertirse, como `"UNKNOWN"` o `"ERROR"`, pasen a ser `NaN`. Esto permite continuar con los cálculos, pero también nos obliga a revisar los faltantes resultantes.

Antes de crear nuevas columnas, resulta útil verificar cuántos valores faltantes tenemos en estas variables.

### Punto de control

Las columnas nuevas deben conservar una relación clara con sus variables de origen. Documentar esa relación ayuda a interpretar los resultados y a repetir el procedimiento.

In [ ]:
df_trabajo[columnas_numericas].isna().sum()

Esta revisión nos recuerda que convertir tipos no elimina los problemas del dataset. Algunos valores quedan como faltantes porque no pudieron interpretarse como números.

Aun así, tener estas columnas en formato numérico nos permite crear nuevas variables calculadas y verificar relaciones internas.

### Punto de control

Trabajar sobre una copia mantiene disponible la fuente original. Así es posible comparar el estado inicial con la versión preparada y revisar cada decisión.

## Generar un importe calculado

Una vez que `quantity` y `price_per_unit` están en formato numérico, es posible crear una nueva columna a partir de ellas.

En un dataset de ventas, una relación esperada es:

```text
importe calculado = cantidad × precio unitario
```

En nuestro caso, trabajaremos con crear una columna llamada `calculated_total`.

Esta columna representa el total que se obtiene al multiplicar la cantidad vendida por el precio unitario.

### Punto de control

Una conversión puede producir valores faltantes cuando encuentra textos especiales. Esos casos deben conservarse visibles para no confundir una ausencia con un dato válido.

In [ ]:
df_trabajo["calculated_total"] = (
    df_trabajo["quantity"] * df_trabajo["price_per_unit"]
)

df_trabajo[
    [
        "quantity",
        "price_per_unit",
        "total_spent",
        "calculated_total"
    ]
].head(10)

La nueva columna `calculated_total` no estaba en el dataset original. La construimos a partir de columnas existentes.

Este tipo de columna puede servir para enriquecer el análisis, pero también para validar datos. Para este ejemplo, el dataset ya tenía una columna llamada `total_spent`, que representa el total registrado de la transacción.

Entonces es posible comparar dos valores:

```text
total_spent        → total registrado en el archivo
calculated_total   → total calculado a partir de cantidad y precio unitario
```

Si ambos valores coinciden, la transacción parece consistente desde el punto de vista del importe. Si no coinciden, puede haber un error en alguna de las columnas o un problema de carga.

Esta es una idea muy relevante: crear columnas nuevas no solo permite agregar información, también puede ayudar a controlar la calidad del dataset.

### Punto de control

Las reglas de validación deben contemplar valores completos y casos insuficientes. No contar con todos los componentes no equivale necesariamente a que exista una inconsistencia.

## Añadir una marca de consistencia

La columna `calculated_total` nos permite comparar el importe registrado con el importe calculado.

Para eso es posible crear una nueva columna booleana llamada `total_matches`.

Esta columna indicará si `total_spent` coincide con `calculated_total`.

### Punto de control

Las variables derivadas amplían las posibilidades de análisis, pero no sustituyen las columnas originales. Mantener ambas versiones permite rastrear cómo se obtuvo cada resultado.

In [ ]:
df_trabajo["total_matches"] = (
    df_trabajo["total_spent"] == df_trabajo["calculated_total"]
)

df_trabajo[
    [
        "quantity",
        "price_per_unit",
        "total_spent",
        "calculated_total",
        "total_matches"
    ]
].head(10)

La columna `total_matches` contiene valores `True` o `False`.

`True` significa que el total registrado coincide con el total calculado. `False` significa que no coinciden.

Aun así, debemos interpretar esta columna con cuidado. Si alguna de las columnas necesarias para el cálculo tiene un valor faltante, el resultado de la comparación puede ser `False`, aunque no necesariamente exista una inconsistencia real. Puede ocurrir simplemente que no tengamos suficiente información para validar esa fila.

Por esa razón, muchas veces resulta útil distinguir entre tres situaciones:

```text
el total coincide
el total no coincide
no hay datos suficientes para comparar
```

Vamos a crear una versión más expresiva de esta validación en la siguiente sección.

### Punto de control

Antes de interpretar una nueva columna, conviene revisar sus tipos, ejemplos y valores frecuentes. Una inspección pequeña puede detectar errores antes de usarlos en cálculos posteriores.

## Clasificar el resultado de la comparación

La columna `total_matches` nos da una primera validación, pero es demasiado simple.

Cuando devuelve `False`, pueden estar ocurriendo dos cosas distintas. Tal vez `total_spent` y `calculated_total` realmente no coinciden. Pero también puede pasar que falte alguno de los datos necesarios para hacer la comparación.

Por ejemplo, si falta `quantity`, `price_per_unit` o `total_spent`, no tenemos información suficiente para validar esa fila.

Para evitar confusiones, es posible crear una columna más descriptiva llamada `estado_total`.

### Punto de control

La verificación final debe confirmar nombres, tipos, dimensiones y ejemplos. Un dataset preparado es aquel cuyo contenido y estructura pueden explicarse con claridad.

In [ ]:
df_trabajo["estado_total"] = "sin_datos_suficientes"

condicion_datos_completos = (
    df_trabajo["quantity"].notna()
    & df_trabajo["price_per_unit"].notna()
    & df_trabajo["total_spent"].notna()
)

condicion_coincide = (
    condicion_datos_completos
    & (df_trabajo["total_spent"] == df_trabajo["calculated_total"])
)

condicion_no_coincide = (
    condicion_datos_completos
    & (df_trabajo["total_spent"] != df_trabajo["calculated_total"])
)

df_trabajo.loc[condicion_coincide, "estado_total"] = "coincide"
df_trabajo.loc[condicion_no_coincide, "estado_total"] = "no_coincide"

df_trabajo[
    [
        "quantity",
        "price_per_unit",
        "total_spent",
        "calculated_total",
        "estado_total"
    ]
].head(15)

A continuación la validación es más clara.

La columna `estado_total` puede tomar tres valores:

```text
coincide
no_coincide
sin_datos_suficientes
```

Esto evita interpretar como error una fila que simplemente no tiene datos suficientes para ser evaluada.

Podemos revisar cuántos casos hay en cada grupo.

### Punto de control

La consistencia de los nombres reduce errores al escribir transformaciones y facilita que otra persona entienda el flujo sin consultar cada línea de código.

In [ ]:
df_trabajo["estado_total"].value_counts(dropna=False)

Este conteo nos permite evaluar la consistencia interna del dataset. En la salida aparecen casos `coincide` y `sin_datos_suficientes`, pero no aparecen casos `no_coincide`. Eso indica que, dentro de las filas que tienen los datos necesarios para hacer la comparación, el total registrado coincide con el total calculado.

Las filas clasificadas como `sin_datos_suficientes` no indican una diferencia entre importes, sino falta de información para poder validar la relación.

Si hubiese muchas filas en `no_coincide`, deberíamos revisar si existe un problema en `quantity`, `price_per_unit`, `total_spent` o en la forma en que fue registrada la transacción.

Si hay muchas filas en `sin_datos_suficientes`, el problema principal no es una diferencia entre importes, sino la falta de información necesaria para validar.

Este tipo de columna es muy útil porque convierte una comparación técnica en una señal más interpretable para el análisis.

### Punto de control

Las columnas nuevas deben conservar una relación clara con sus variables de origen. Documentar esa relación ayuda a interpretar los resultados y a repetir el procedimiento.

## Derivar variables desde una fecha

Además de crear columnas numéricas y columnas de validación, también es posible crear nuevas columnas a partir de una fecha.

En nuestro dataset, la columna `transaction_date` todavía necesita convertirse a formato temporal. Una vez convertida, podremos extraer información como el año, el mes o el día de la semana.

Primero convertimos la columna:

### Punto de control

Trabajar sobre una copia mantiene disponible la fuente original. Así es posible comparar el estado inicial con la versión preparada y revisar cada decisión.

In [ ]:
df_trabajo["transaction_date"] = pd.to_datetime(
    df_trabajo["transaction_date"],
    errors="coerce"
)

df_trabajo["transaction_date"].dtype

A continuación `transaction_date` está en formato temporal.

Podemos crear algunas columnas derivadas:

### Punto de control

Una conversión puede producir valores faltantes cuando encuentra textos especiales. Esos casos deben conservarse visibles para no confundir una ausencia con un dato válido.

In [ ]:
df_trabajo["year"] = df_trabajo["transaction_date"].dt.year
df_trabajo["month"] = df_trabajo["transaction_date"].dt.month
df_trabajo["day_of_week"] = df_trabajo["transaction_date"].dt.day_name()

df_trabajo[
    [
        "transaction_date",
        "year",
        "month",
        "day_of_week"
    ]
].head(10)

Estas columnas derivadas permiten analizar las transacciones desde una perspectiva temporal.

La columna `year` indica el año de la transacción. La columna `month` indica el número de mes. La columna `day_of_week` indica el día de la semana.

Observemos que `year` y `month` pueden aparecer con decimales, como `2023.0` o `9.0`. Esto ocurre porque algunas fechas no pudieron convertirse y quedaron como `NaT`. Al derivar columnas desde fechas faltantes, Pandas necesita usar un formato que también pueda representar esos valores ausentes.

También vemos que los nombres de los días aparecen en inglés, como `Friday`, `Tuesday` o `Wednesday`. Esto depende del entorno y no afecta el análisis: los valores siguen representando correctamente el día de la semana. Si más adelante necesitáramos presentar esos nombres en español, podríamos aplicar un reemplazo o mapeo de valores.

Al igual que con las columnas numéricas, estas nuevas variables no estaban explícitas en el dataset original. Las construimos a partir de una columna existente.

Crear columnas temporales puede ser útil para analizar períodos, detectar patrones por mes o estudiar diferencias entre días de la semana.

Si algunas filas tienen fechas faltantes o no interpretables, las columnas derivadas también quedarán con valores faltantes. Por esa razón, cada nueva columna debe interpretarse teniendo en cuenta la calidad de la columna original.

### Punto de control

Las reglas de validación deben contemplar valores completos y casos insuficientes. No contar con todos los componentes no equivale necesariamente a que exista una inconsistencia.

## Revisar la estructura resultante

Después de renombrar columnas, convertir tipos y crear nuevas variables, resulta útil revisar cómo quedó el `DataFrame`.

La verificación nos ayuda a confirmar que las transformaciones se aplicaron correctamente y que las nuevas columnas tienen sentido.

Podemos comenzar revisando los nombres de columnas actuales.

### Punto de control

Las variables derivadas amplían las posibilidades de análisis, pero no sustituyen las columnas originales. Mantener ambas versiones permite rastrear cómo se obtuvo cada resultado.

In [ ]:
df_trabajo.columns

A continuación el `DataFrame` contiene tanto columnas originales renombradas como columnas nuevas creadas durante el proceso.

Entre las nuevas columnas tenemos:

```text
calculated_total
total_matches
estado_total
year
month
day_of_week
```

También es posible revisar una vista general de algunas columnas relevantes.

### Punto de control

Antes de interpretar una nueva columna, conviene revisar sus tipos, ejemplos y valores frecuentes. Una inspección pequeña puede detectar errores antes de usarlos en cálculos posteriores.

In [ ]:
df_trabajo[
    [
        "transaction_id",
        "item",
        "quantity",
        "price_per_unit",
        "total_spent",
        "calculated_total",
        "estado_total",
        "transaction_date",
        "year",
        "month",
        "day_of_week"
    ]
].head(10)

Esta vista permite observar, en una misma tabla, las variables originales ya preparadas y las nuevas columnas derivadas.

También es posible revisar los tipos de datos:

### Punto de control

La verificación final debe confirmar nombres, tipos, dimensiones y ejemplos. Un dataset preparado es aquel cuyo contenido y estructura pueden explicarse con claridad.

In [ ]:
df_trabajo.info()

Esta revisión final es relevante porque el `DataFrame` cambió bastante desde la carga inicial.

Primero estandarizamos nombres de columnas. Luego convertimos columnas numéricas. Después creamos columnas calculadas y columnas de validación. Finalmente convertimos la fecha y generamos columnas temporales.

Cada una de esas transformaciones modifica la estructura del dataset. Por esa razón, después de crear nuevas columnas, resulta útil revisar tanto los nombres como los tipos de datos y algunos ejemplos concretos.

Preparar datos no es solo transformar. También es verificar que cada transformación haya dejado el dataset en mejores condiciones para el análisis.

### Punto de control

La consistencia de los nombres reduce errores al escribir transformaciones y facilita que otra persona entienda el flujo sin consultar cada línea de código.

## Precauciones durante la preparación

Al estandarizar nombres de columnas, uno de los errores más frecuentes es cambiar nombres sin verificar después la estructura del `DataFrame`.

Si renombramos una columna y luego intentamos usar el nombre anterior, Pandas no va a encontrarla. Por esa razón, después de aplicar `rename()`, resulta útil revisar `df.columns` y asegurarnos de usar los nombres nuevos en las celdas siguientes.

Otro error posible es crear nuevas columnas antes de preparar los tipos de datos. Por ejemplo, si `quantity` o `price_per_unit` todavía estuvieran cargadas como texto, la columna `calculated_total` no sería confiable. Antes de crear variables numéricas derivadas, debemos convertir las columnas necesarias y revisar si aparecieron nuevos faltantes.

También debemos tener cuidado con las columnas de validación. Una comparación como:

```python
df_trabajo["total_spent"] == df_trabajo["calculated_total"]
```

puede devolver `False` cuando falta algún dato necesario para comparar. Eso no siempre significa que haya una inconsistencia real. Por esa razón creamos `estado_total`, que distingue entre coincidencia, diferencia y falta de datos suficientes.

Otro error frecuente es crear muchas columnas nuevas sin un propósito claro. Cada nueva variable debería tener una función: enriquecer el análisis, simplificar una consulta, validar otra columna o preparar el dataset para una etapa posterior.

Finalmente, al crear columnas derivadas de fechas, debemos recordar que dependen de la calidad de la fecha original. Si `transaction_date` no pudo convertirse correctamente y quedó como `NaT`, las columnas derivadas como `year`, `month` o `day_of_week` también quedarán sin información válida.

Una buena rutina para esta etapa podría ser:

```text
renombrar columnas con un criterio claro
verificar los nombres nuevos
convertir columnas necesarias antes de calcular
crear columnas derivadas con un propósito definido
crear columnas de validación cuando haya relaciones internas
verificar tipos, faltantes y ejemplos concretos
```

Estandarizar nombres y crear columnas nuevas no son pasos aislados. Forman parte de un proceso de preparación que busca dejar el dataset más claro, más consistente y más útil para el análisis.

### Punto de control

Las columnas nuevas deben conservar una relación clara con sus variables de origen. Documentar esa relación ayuda a interpretar los resultados y a repetir el procedimiento.

## Síntesis del trabajo

En esta práctica trabajamos con dos tareas relevantes dentro de la preparación de datos: estandarizar nombres de columnas y crear nuevas columnas útiles para el análisis.

Primero retomamos la importancia de los nombres de columnas. El dataset original tenía nombres como `Transaction ID`, `Price Per Unit`, `Total Spent`, `Payment Method` y `Transaction Date`. Esos nombres son comprensibles, pero tienen espacios y mayúsculas, lo que puede volver el código menos cómodo de escribir.

Por esa razón creamos una copia del dataset y renombramos las columnas con un criterio uniforme:

```python
df_trabajo = df_trabajo.rename(columns={
    "Transaction ID": "transaction_id",
    "Item": "item",
    "Quantity": "quantity",
    "Price Per Unit": "price_per_unit",
    "Total Spent": "total_spent",
    "Payment Method": "payment_method",
    "Location": "location",
    "Transaction Date": "transaction_date"
})
```

El criterio elegido fue usar nombres en minúsculas, sin espacios y con guiones bajos. Este estilo facilita la escritura del código y ayuda a mantener una estructura más consistente.

Después preparamos algunas columnas antes de crear nuevas variables. Convertimos `quantity`, `price_per_unit` y `total_spent` a formato numérico usando `pd.to_numeric()` con `errors="coerce"`:

```python
for columna in columnas_numericas:
    df_trabajo[columna] = pd.to_numeric(
        df_trabajo[columna],
        errors="coerce"
    )
```

Este paso fue necesario porque no resulta útil crear columnas calculadas a partir de valores que todavía están cargados como texto.

Luego creamos una nueva columna llamada `calculated_total`:

```python
df_trabajo["calculated_total"] = (
    df_trabajo["quantity"] * df_trabajo["price_per_unit"]
)
```

Esta columna representa el total calculado a partir de la cantidad y el precio unitario.

Como el dataset ya tenía una columna `total_spent`, pudimos comparar el total registrado con el total calculado. Primero creamos una validación simple llamada `total_matches`, pero luego vimos que esa comparación podía ser insuficiente cuando faltaban datos.

Por esa razón creamos una columna más expresiva, `estado_total`, con tres estados posibles:

```text
coincide
no_coincide
sin_datos_suficientes
```

Esta columna permitió distinguir entre una inconsistencia real y una fila que no tenía datos suficientes para ser evaluada.

También convertimos `transaction_date` a formato temporal y creamos columnas derivadas de la fecha:

```python
df_trabajo["year"] = df_trabajo["transaction_date"].dt.year
df_trabajo["month"] = df_trabajo["transaction_date"].dt.month
df_trabajo["day_of_week"] = df_trabajo["transaction_date"].dt.day_name()
```

Estas columnas permiten analizar las transacciones desde una perspectiva temporal.

Finalmente verificamos las columnas creadas, los tipos de datos y algunos ejemplos concretos del `DataFrame` transformado.

La idea principal de esta sección fue:

```text
Estandarizar nombres y crear columnas nuevas ayuda a transformar un dataset crudo en una tabla más clara, verificable y útil para el análisis.
```

Crear columnas no es solo agregar información. También puede servir para validar relaciones internas, detectar inconsistencias y preparar el dataset para análisis posteriores.

### Punto de control

Trabajar sobre una copia mantiene disponible la fuente original. Así es posible comparar el estado inicial con la versión preparada y revisar cada decisión.